In [47]:
from spells import summon, ColName, ColType, ColSpec
from spells.columns import agg_col

import polars as pl

sets = ["EOE", "FIN", "TDM", "DFT"]

df = summon(
    sets,
    [
        ColName.ATA, 
        ColName.GIH_WR,
        ColName.NUM_GIH
    ],
    group_by=[
        ColName.NAME,
        ColName.EXPANSION,
        ColName.FORMAT_DAY
    ],
    filter_spec={
        "lhs": "format_day",
        "op": ">",
        "rhs": 12
    }
).filter(
    (pl.col(ColName.NUM_GIH)>100)
)

prev_df=df.select(
    "expansion", 
    "name",
    (pl.col("format_day")-1).alias("prev_day"),
    pl.col("ata").alias("ata_next"),
    pl.col("gih_wr").alias("gih_wr_next")
)

join_df=df.join(
    prev_df, 
    left_on=["expansion", "name", "format_day"],
    right_on=["expansion", "name", "prev_day"]
).with_columns(
    (pl.col("gih_wr_next")-pl.col("gih_wr")
    ).alias("gih_del"),
    (pl.col("ata_next")-pl.col("ata")
    ).alias("ata_del"),
).select(
    "expansion",
    "name",
    "format_day",
    "gih_del",
    "ata_del",
    "ata"
)

In [49]:
join_df.shape

(22051, 6)

In [51]:
import numpy as np

sol=np.linalg.lstsq(
    join_df.select(
        "ata_del", 
        pl.col("ata")*pl.col("ata_del")
    ),
    join_df["gih_del"],
)
sol



# change in GIH WR per unit change
# in ATA day-to-day after
# format day 12, 2025 sets

LinAlgError: SVD did not converge in Linear Least Squares